# Two-Material Scenario

This notebook shows how to:
1. Load two materials from the pvdeg material databases
2. Bind a different degradation function to each material layer
3. Run the pipeline and inspect the results

In [16]:
import pvdeg
import os
import pandas as pd
import json

## Load weather data

Read a local PSM4 weather file that ships with the pvdeg tutorials.

In [17]:
weather_df = pd.read_csv("../data/psm4_golden.csv", index_col=0, parse_dates=True)
with open("../data/meta_golden.json", "r") as f:
    meta = json.load(f)

## Create the scenario and load two materials

Pass `materials` as a dict to assign a named layer to each material.
Each entry points to a material key in one of the pvdeg JSON databases
(`"O2permeation"`, `"H2Opermeation"`, or `"AApermeation"`).

Here we load **OX003** (EVA encapsulant) and **OX004** (another encapsulant variant)
from `O2permeation.json` as two separate layers.

In [18]:
scenario = pvdeg.Scenario(
    name="two-material-demo",
    weather_data=weather_df,
    meta_data=meta,
)

scenario.addModule(
    module_name="glass-polymer",
    materials={
        "encapsulant": {
            "material_file": "O2permeation",
            "material_name": "OX003",
        },
        "backsheet": {
            "material_file": "O2permeation",
            "material_name": "OX004",
        },
    },
)

## Add one degradation job per material layer

Use a 2-tuple `(function, layer_name)` so that `run()` knows which material
parameters to inject for each job.

- **IwaVantHoff** characterises the outdoor degradation environment for the encapsulant
- **standoff** computes the minimum mounting standoff for the backsheet side

Material parameters from the database that do not appear in a function's
signature (e.g. `Ead`, `Do`) are silently filtered out, so you don't need
to worry about parameter mismatches.

In [19]:
scenario.addJob(func=(pvdeg.degradation.IwaVantHoff, "encapsulant"))
scenario.addJob(func=(pvdeg.degradation.arrhenius, "backsheet"))

## Run the pipeline

In [20]:
scenario.run()

The array surface_tilt angle was not provided, therefore the latitude of  39.7 was used.
The array azimuth was not provided, therefore an azimuth of  180.0 was used.


## Inspect results

`results` is a nested dict: `results[module_name][job_id]`.
Use `display(scenario)` to see job IDs, then index into them directly.

In [21]:
display(scenario)

,Year,Month,Day,Hour,Minute,temp_air,dew_point,dhi,dni,ghi,albedo,pressure,wind_direction,wind_speed,relative_humidity
0,2011,1,1,0,30,-16.0,-26.6,0.0,0.0,0.0,0.8,794.0,276.0,4.0,39.692741
1,2011,1,1,1,30,-16.0,-26.5,0.0,0.0,0.0,0.8,794.0,274.0,4.1,40.057183
2,2011,1,1,2,30,-15.9,-26.2,0.0,0.0,0.0,0.8,795.0,273.0,4.1,40.828082
3,2011,1,1,3,30,-15.8,-26.0,0.0,0.0,0.0,0.8,795.0,272.0,4.1,41.234476
4,2011,1,1,4,30,-15.7,-25.7,0.0,0.0,0.0,0.8,795.0,271.0,4.0,42.023356
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2003,12,31,19,30,-15.7,-26.8,0.0,0.0,0.0,0.8,793.0,284.0,3.9,38.014453
8756,2003,12,31,20,30,-15.6,-27.0,0.0,0.0,0.0,0.8,793.0,282.0,4.0,37.015745
8757,2003,12,31,21,30,-15.5,-26.8,0.0,0.0,0.0,0.8,793.0,281.0,4.0,37.390056
8758,2003,12,31,22,30,-15.5,-26.6,0.0,0.0,0.0,0.8,794.0,280.0,3.9,38.080774


In [22]:
module_results = scenario.results["glass-polymer"]

for job_id, result in module_results.items():
    print(f"job {job_id}: {result}")

job GZPXO: 233.98942611077612
job OGPBK: 8760.0
